# Homework 7.2 - Task 3

## Marcus-Conditioned Odyssey-Style Generation

This notebook generates short texts from the statistical Odyssey model.

The base model is unchanged across themes.

Conditioning is introduced through a fixed multiplicative probability bias toward the predeclared theme words.

## 1. Base generation distribution

Let $P_0(w|u,v)$ denote the interpolated Odyssey distribution based on:

- trigram context $(u,v)$;
- bigram context $v$;
- the unigram distribution.

The interpolation weights are

$$
0.75,\ 0.20,\ 0.05.
$$

The condition does not replace the base language model.

## 2. Conditioning rule

For theme $c$, let $B_c$ be its fixed rewarded-word set.

The conditioned distribution is proportional to

$$
P_0(w|u,v)
\exp(\lambda)
$$

when $w$ is in $B_c$, and proportional to

$$
P_0(w|u,v)
$$

otherwise.

The fixed conditioning strength is

$$
\lambda = 1.5.
$$

This is implemented exactly by rejection sampling from the base model.

The rule deliberately rewards surface theme vocabulary. It does not guarantee preservation of the ethical stance of the selected Marcus passage.

## 3. Sampling plan

For each of the five themes, generate:

- 500 unconditioned samples;
- 500 conditioned samples.

Therefore the total number of generated texts is

$$
5 \times 2 \times 500 = 5000.
$$

The base seed is fixed in Task 1.

Generation stops when the model emits the end-of-sentence marker or when 30 words have been produced.

A sentence stopped by the 30-word limit is explicitly marked as truncated.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math
import pickle

import numpy as np
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw7_2"

design = json.loads(
    (
        DATA_DIR
        / "task1_fixed_experiment_design.json"
    ).read_text(
        encoding="utf-8"
    )
)

with open(
    DATA_DIR
    / "task2_odyssey_ngram_model.pkl",
    "rb",
) as handle:
    model = pickle.load(
        handle
    )

unigram = model[
    "unigram"
]

bigram = model[
    "bigram"
]

trigram = model[
    "trigram"
]

rewarded_words = model[
    "rewarded_words_in_vocab"
]

probe_words = model[
    "probe_words_in_vocab"
]

weights = model[
    "interpolation_weights"
]

LAMBDA = float(
    design[
        "conditioning_strength_lambda"
    ]
)

MAX_WORDS = int(
    design[
        "max_generated_words"
    ]
)

N_DRAWS = int(
    design[
        "draws_per_theme_per_mode"
    ]
)

BASE_SEED = int(
    design[
        "base_generation_seed"
    ]
)

THEMES = design[
    "themes"
]

print("Themes:", THEMES)
print("Draws per theme per mode:", N_DRAWS)
print("Conditioning strength:", LAMBDA)

Themes: ['discipline', 'mortality', 'anger', 'duty', 'self-command']
Draws per theme per mode: 500
Conditioning strength: 1.5


## 4. Efficient exact sampler

The base interpolation is sampled as a mixture.

At each step:

1. choose trigram, bigram, or unigram according to the available normalized mixture weights;
2. sample from that empirical count distribution.

To obtain the multiplicatively biased conditioned distribution, propose from the base distribution and use rejection sampling.

Theme words are always accepted.

Non-theme words are accepted with probability

$$
\exp(-\lambda).
$$

This produces the required multiplicative theme bias without evaluating a full vocabulary-sized probability vector at every step.

In [2]:
counter_sampler_cache = {}


def cached_counter_arrays(
    counter,
    cache_key,
):
    if cache_key in counter_sampler_cache:
        return counter_sampler_cache[
            cache_key
        ]

    items = np.array(
        list(
            counter.keys()
        ),
        dtype=object,
    )

    counts = np.array(
        list(
            counter.values()
        ),
        dtype=float,
    )

    cumulative = np.cumsum(
        counts
    )

    counter_sampler_cache[
        cache_key
    ] = (
        items,
        cumulative,
    )

    return items, cumulative


def sample_counter(
    counter,
    rng,
    cache_key,
):
    items, cumulative = (
        cached_counter_arrays(
            counter,
            cache_key,
        )
    )

    value = rng.random() * cumulative[-1]

    index = np.searchsorted(
        cumulative,
        value,
        side="right",
    )

    return str(
        items[index]
    )


def sample_base_word(
    previous_two,
    previous_one,
    rng,
):
    components = []
    component_weights = []

    if previous_two in trigram:
        components.append(
            (
                "trigram",
                previous_two,
                trigram[
                    previous_two
                ],
            )
        )

        component_weights.append(
            weights["trigram"]
        )

    if previous_one in bigram:
        components.append(
            (
                "bigram",
                previous_one,
                bigram[
                    previous_one
                ],
            )
        )

        component_weights.append(
            weights["bigram"]
        )

    components.append(
        (
            "unigram",
            "GLOBAL",
            unigram,
        )
    )

    component_weights.append(
        weights["unigram"]
    )

    component_weights = np.array(
        component_weights,
        dtype=float,
    )

    component_weights = (
        component_weights
        / component_weights.sum()
    )

    component_index = int(
        rng.choice(
            len(components),
            p=component_weights,
        )
    )

    component_name, context, counter = (
        components[
            component_index
        ]
    )

    return sample_counter(
        counter,
        rng,
        cache_key=(
            component_name,
            context,
        ),
    )


def sample_conditioned_word(
    previous_two,
    previous_one,
    theme,
    rng,
):
    reward_set = set(
        rewarded_words[
            theme
        ]
    )

    nonreward_acceptance = math.exp(
        -LAMBDA
    )

    while True:
        proposal = sample_base_word(
            previous_two,
            previous_one,
            rng,
        )

        if proposal in reward_set:
            return proposal

        if (
            rng.random()
            < nonreward_acceptance
        ):
            return proposal

In [3]:
def generate_sentence(
    theme,
    conditioned,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    history = [
        "<s>",
        "<s>",
    ]

    generated = []
    stopped_by_end = False

    for _ in range(
        MAX_WORDS
    ):
        previous_two = (
            history[-2],
            history[-1],
        )

        previous_one = (
            history[-1]
        )

        if conditioned:
            next_word = (
                sample_conditioned_word(
                    previous_two,
                    previous_one,
                    theme,
                    rng,
                )
            )

        else:
            next_word = (
                sample_base_word(
                    previous_two,
                    previous_one,
                    rng,
                )
            )

        if next_word == "</s>":
            stopped_by_end = True
            break

        generated.append(
            next_word
        )

        history.append(
            next_word
        )

    truncated = (
        not stopped_by_end
        and len(generated)
        >= MAX_WORDS
    )

    if generated:
        text = (
            " ".join(
                generated
            )
        )

        text = (
            text[0].upper()
            + text[1:]
            + "."
        )

    else:
        text = "."

    return {
        "text": text,
        "tokens": generated,
        "length": len(
            generated
        ),
        "truncated": (
            truncated
        ),
        "stopped_by_end_marker": (
            stopped_by_end
        ),
    }

## 5. Generate all 5000 samples

In [4]:
generation_rows = []

for theme_index, theme in enumerate(
    THEMES
):
    for mode_index, conditioned in enumerate(
        [
            False,
            True,
        ]
    ):
        mode = (
            "conditioned"
            if conditioned
            else "unconditioned"
        )

        for sample_index in range(
            N_DRAWS
        ):
            seed = (
                BASE_SEED
                + 100000
                * theme_index
                + 10000
                * mode_index
                + sample_index
            )

            result = generate_sentence(
                theme=theme,
                conditioned=conditioned,
                seed=seed,
            )

            token_set = result[
                "tokens"
            ]

            reward_set = set(
                rewarded_words[
                    theme
                ]
            )

            probe_set = set(
                probe_words[
                    theme
                ]
            )

            reward_hits = sum(
                word in reward_set
                for word
                in token_set
            )

            probe_hits = sum(
                word in probe_set
                for word
                in token_set
            )

            generation_rows.append({
                "theme": theme,
                "mode": mode,
                "sample_index": (
                    sample_index
                ),
                "seed": seed,
                "text": result[
                    "text"
                ],
                "length": result[
                    "length"
                ],
                "truncated": result[
                    "truncated"
                ],
                "stopped_by_end_marker": (
                    result[
                        "stopped_by_end_marker"
                    ]
                ),
                "reward_hits": (
                    reward_hits
                ),
                "probe_hits": (
                    probe_hits
                ),
                "reward_rate_per_100_words": (
                    100.0
                    * reward_hits
                    / max(
                        1,
                        result[
                            "length"
                        ],
                    )
                ),
                "probe_rate_per_100_words": (
                    100.0
                    * probe_hits
                    / max(
                        1,
                        result[
                            "length"
                        ],
                    )
                ),
            })

        print(
            "Finished:",
            theme,
            mode,
        )


generations = pd.DataFrame(
    generation_rows
)

generation_path = (
    DATA_DIR
    / "task3_statistical_generations.csv"
)

generations.to_csv(
    generation_path,
    index=False,
)

print()
print(
    generations.groupby(
        [
            "theme",
            "mode",
        ]
    ).size()
)

print()
print("Saved:", generation_path.resolve())

Finished: discipline unconditioned
Finished: discipline conditioned
Finished: mortality unconditioned
Finished: mortality conditioned
Finished: anger unconditioned
Finished: anger conditioned
Finished: duty unconditioned
Finished: duty conditioned
Finished: self-command unconditioned
Finished: self-command conditioned

theme         mode         
anger         conditioned      500
              unconditioned    500
discipline    conditioned      500
              unconditioned    500
duty          conditioned      500
              unconditioned    500
mortality     conditioned      500
              unconditioned    500
self-command  conditioned      500
              unconditioned    500
dtype: int64

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2\data\hw7_2\task3_statistical_generations.csv


## 6. Record the fixed-rule displayed examples

The displayed examples are selected by the rule frozen in Task 1.

For each theme and mode:

- choose the smallest sample index with 8 to 30 words and not truncated;
- if none exists, use the smallest sample index.

This prevents favorable example selection after seeing the outputs.

In [5]:
display_rows = []

for theme in THEMES:
    for mode in [
        "unconditioned",
        "conditioned",
    ]:
        subset = (
            generations[
                (
                    generations[
                        "theme"
                    ]
                    == theme
                )
                & (
                    generations[
                        "mode"
                    ]
                    == mode
                )
            ]
            .sort_values(
                "sample_index"
            )
        )

        eligible = subset[
            (
                subset[
                    "length"
                ]
                >= 8
            )
            & (
                subset[
                    "length"
                ]
                <= 30
            )
            & (
                ~subset[
                    "truncated"
                ]
            )
        ]

        selected = (
            eligible.iloc[0]
            if len(
                eligible
            )
            else subset.iloc[0]
        )

        display_rows.append(
            selected.to_dict()
        )


display_examples = pd.DataFrame(
    display_rows
)

display_path = (
    DATA_DIR
    / "task3_fixed_display_examples.csv"
)

display_examples.to_csv(
    display_path,
    index=False,
)

print(
    display_examples[
        [
            "theme",
            "mode",
            "sample_index",
            "seed",
            "length",
            "truncated",
            "text",
        ]
    ].to_string(
        index=False
    )
)

       theme          mode  sample_index   seed  length  truncated                                                                                                                                 text
  discipline unconditioned             3  63875      12      False                                                                             I have left at pylos are already urging her to the town.
  discipline   conditioned             0  73872      24      False                     It was that of ajax who was as fast as he leaned upon his native land showed on the plain was filled with blood.
   mortality unconditioned             6 163878      13      False                                                                    He was saying and agreed that i father i have twenty geese about.
   mortality   conditioned             1 173873      17      False                                                         Surely no man and his acceptance let us a mischief but i was in the house a.


# Task 3 Conclusion

The same statistical Odyssey model has now produced unconditioned and Marcus-theme-conditioned samples.

The conditioned samples differ only through the fixed word-probability bias chosen before generation.

Task 4 will compare:

- the fixed-rule unconditioned output;
- the fixed-rule conditioned output;
- one direct LLM-generated sentence for the same theme.